# 🚧 IN PROGRESING ..... 🚧

## First Semester:

**🤷🏾‍♂️ What will I do?**
* **Problem Definition**
* **Literature Review**
* **[GitHub Setup](https://github.com/CCT-Dublin/CCT_ST_CA_01_20S)**
* **Exploratory Data Analysis (EDA)**
* **Initial Pre-processing.**

### 🏦 Business Problem Definition

**Information about the selected dataset**

* **[Bitcoin Heist Ransomware Address](https://archive.ics.uci.edu/dataset/526/bitcoinheistransomwareaddressdataset):**
We have downloaded and parsed the entire Bitcoin transaction graph from 2009 January to 2018 December. Using a time interval of 24 hours, we extracted daily transactions on the network and formed the Bitcoin graph. We filtered out the network edges that transfer less than B0.3, since ransom amounts are rarely below this threshold.

Ransomware addresses are taken from three widely adopted studies: Montreal, Princeton and Padua. Please see the BitcoinHeist article for references.

**🚨 Problem Definition and Motivation**

* **Domain: Business (Cryptocurrency / Financial Crime Risk & Compliance)**
* **Problem Type:** Supervised Machine Learning – Binary Classification
* **Target Variable ($Y$):** Ransomware / Illicit Bitcoin Address (`Illicit / Ransomware` = 1 | `White / Legitimate` = 0)

&gt; Ransomware attacks and cryptocurrency-related fraud pose severe financial, legal, and operational risks to digital asset exchanges, financial institutions, and retail investors. When malicious actors execute ransomware campaigns and demand cryptocurrency payments, they leverage blockchain anonymity to launder funds across complex networks, leading to massive financial losses, regulatory non-compliance (Anti-Money Laundering / KYC violations), and severe reputational damage.

&gt; Traditionally, financial fraud monitoring relied on static rules and manual address blacklists; however, these approaches fail to dynamically capture sophisticated blockchain laundering techniques, such as coin mixing, multi-path splitting, and address merging. The selected dataset from the UCI Machine Learning Repository (**BitcoinHeist Ransomware Address Dataset**) addresses this challenge by tracking daily Bitcoin network address transactions, capturing specialized graph-based features (`length`, `weight`, `count`, `looped`, `neighbors`, and `income`) designed to quantify complex transactional patterns across multiple ransomware families (e.g., CryptoLocker, CryptXXX).

&gt; Data analytics and predictive modelling offer a strategic business solution to this problem by enabling automated, real-time detection of illicit cryptocurrency addresses. By leveraging supervised machine learning algorithms on historical graph transaction patterns, compliance teams and financial institutions can proactively identify high-risk wallets, flag suspicious money laundering behavior before fiat off-ramping, ensure regulatory compliance, and mitigate financial crime risks.

**🎯 Objectives:**

* **Data Pre-processing &amp; Class Imbalance Handling:** Clean the dataset, encode the target variable (`Ransomware / Illicit` \= 1, `White / Legitimate` \= 0), apply feature scaling on skewed graph metrics, and implement resampling or weighting strategies (e.g., SMOTE, class weighting) to address severe class imbalance.
* **Model Development:** Implement at least two distinct supervised classification algorithms (**[XGBoost](https://xgboost.readthedocs.io/en/stable/)** and **[Random Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)**) to predict illicit transaction probabilities based on network graph behavior.
* **Multi-Split Performance Evaluation:** Evaluate model accuracy and stability across distinct training set sizes (e.g., **15%**, and **30%**) using **Cross-Validation** techniques to analyze how restricted training data impacts learning stability and variance.
* **Hyperparameter Optimization:** Apply `GridSearchCV` on selected model pipelines to systematically tune key hyperparameters (e.g., `learning_rate`, `colsample_bytree`, `max_depth` for XGBoost \~\~ `n_estimators`, `max_depth` for Random Forest).
* **Comprehensive Metric Assessment:** Evaluate predictions using a full suite of classification metrics, including **Accuracy**, **Precision**, **Recall (Sensitivity)**, **F1-Score**, **ROC-AUC / PR-AUC**, and **Confusion Matrix**. In a financial crime context, primary emphasis is placed on maximizing **Recall** to minimize critical **False Negatives** (failing to identify illicit ransomware wallets) while maintaining sufficient **Precision** to avoid flagging legitimate users.
* **Comparative Analysis &amp; Feature Importance:** Interpret model outcomes, evaluate feature importance rankings (specifically analyzing the predictive impact of graph features like `length`, `weight`, and `looped`), and discuss trade-offs regarding **right-skewed distributions**, **overfitting**, **underfitting**, and model generalizability.


**📈 Expected Benefits**

* **Financial Crime Mitigation:** Intercept and flag ransomware payouts and laundering attempts before illicit funds are converted into fiat currency.
* **Proactive Regulatory Compliance:** Enable compliance and fraud prevention teams to dynamically flag high-risk wallet addresses, fulfilling Anti-Money Laundering (AML) obligations and reducing exposure to regulatory fines.
* **Automated Blockchain Monitoring:** Standardize and accelerate real-time transaction monitoring across large-scale blockchain networks, reducing manual forensic overhead and operational review costs.

### 🔄 Download the libraries if you don't have them!

In [1]:
#!pip install -r ../requirements.txt

### 🤔 JupyterLab Authoring

In [2]:
%reload_ext watermark
%watermark --author "Matheus DoS S. Silva"
%watermark --email "2026438@student.cct.ie"
%watermark --github_username "MattheuSxS"
%watermark --date
%watermark --python

Author: Matheus DoS S. Silva

Email: 2026438@student.cct.ie

Github username: MattheuSxS

Date: 2026-10-18

Python implementation: CPython
Python version       : 3.14.7
IPython version      : 9.17.1



### 🛂 Importing Packages
This is the workspace preparation phase. Here, we load all the necessary Python tools and libraries (such as Pandas for data manipulation, Scikit-learn for modeling, and Matplotlib/Seaborn for visualizations) that will be used throughout the project.

In [3]:
import math
import logging
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from pathlib import Path
from typing import List, Dict
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.metrics import confusion_matrix, classification_report

# Display settings
sns.set_style('whitegrid')
%matplotlib inline

In [4]:
logging.basicConfig(
    format=("%(asctime)s | %(levelname)s | File_name ~> %(module)s.py "
            "| Function ~> %(funcName)s | Line ~~> %(lineno)d  ~~>  %(message)s"),
    level=logging.INFO
)

### 🔄 Loading and Understanding the Data

This is the first encounter with the project's raw material. We load the data from a source (in our case, a CSV file) into a Pandas DataFrame and perform an initial check to understand its structure.

The download_dataset() function is responsible for loading the dataset used in the Machine Learning project. 
* **1 ~~>** It first checks whether the dataset already exists locally in the ../data directory.
* **2 ~~>** If the file is not found, it attempts to download the dataset from the UC Irvine Machine Learning Repository using the fetch_ucirepo function.
 
Once the dataset is available, the function loads it into a Pandas DataFrame and returns it for further data analysis and Machine Learning processing. The function also includes logging and exception handling to provide information about the download and loading process.

In [5]:
def download_dataset() -> pd.DataFrame:
    data_dir = Path("../data")
    dataset_file = data_dir / "data.csv"

    data_dir.mkdir(parents=True, exist_ok=True)

    try:
        if not dataset_file.exists():

            logging.warning("📥 Dataset not found. Downloading from UC Irvine Machine Learning Repository...")

            import zipfile
            import requests
            from io import BytesIO
            from tqdm.auto import tqdm

            url = "https://archive.ics.uci.edu/static/public/526/bitcoinheistransomwareaddressdataset.zip"
            with requests.get(url, stream=True, timeout=(15, 120)) as response:
                response.raise_for_status()
            
                total_size = int(response.headers.get("content-length", 0))
            
                with BytesIO() as buffer:
                    with tqdm(
                        total = total_size or None,
                        desc = "Downloading ZIP",
                        unit = "B",
                        unit_scale = True,
                        unit_divisor = 1024,
                        bar_format = (
                            "{desc}: {percentage:3.0f}%|{bar}| "
                            "{n_fmt}/{total_fmt} "
                            "[{elapsed}<{remaining}, {rate_fmt}]"
                        ) if total_size else None,
                    ) as progress:
            
                        for chunk in response.iter_content(chunk_size=1024 * 1024):
                            if chunk:
                                buffer.write(chunk)
                                progress.update(len(chunk))
            
                    buffer.seek(0)
                    with zipfile.ZipFile(buffer) as zip_file:
                        csv_files = [
                            name for name in zip_file.namelist()
                            if name.lower().endswith(".csv")
                        ]
            
                        if not csv_files:
                            raise FileNotFoundError(
                                "No CSV file found in the ZIP."
                            )
            
                        with zip_file.open(csv_files[0]) as file:
                            return pd.read_csv(file)

        else:
            logging.info(f"✅ Dataset already exists: {dataset_file}")

        return  pd.read_csv(dataset_file, sep=",")

    except Exception as e:
        logging.error(f"❌ Failed to download or read the dataset: {e}")

In [6]:
df = download_dataset()

2026-10-18 20:44:58,154 | INFO | File_name ~> 3081701666.py | Function ~> download_dataset | Line ~~> 58  ~~>  ✅ Dataset already exists: ../data/data.csv


In [7]:
print(f"Rows: {df.shape[0]:,} ~ Columns: {df.shape[1]}")

Rows: 2,916,697 ~ Columns: 10


**📍 Variable Information:**

* **address:** String    ~~> Bitcoin address.
* **year:** Integer      ~~> Year of observation.
* **day:** Integer       ~~> Day of the year. 1 is the first day, 365 is the last day.
* **length:** Integer    ~~> Measurement of transaction chain length.
* **weight:** Float      ~~> Measure related to the flow of currencies and the structure of transactions
* **count:** Integer     ~~> Count of associated transactions
* **looped:** Integer    ~~> Counting of repeated paths within the transaction structure
* **neighbors:** Integer ~~> Number of associated neighboring transactions 
* **income:** Integer    ~~> Amount received, in satoshis (1 bitcoin = 100 million satoshis).
* **label:** String      ~~> Name of the ransomware family (e.g., Cryptxxx, cryptolocker etc) or white (i.e., not known to be ransomware).

In [8]:
df.head()

,address,year,day,length,weight,count,looped,neighbors,income,label
0,111K8kZAEnJg245r2cM6y9zgJGHZtJPy6,2017,11,18,0.008333,1,0,2,100050000.0,princetonCerber
1,1123pJv8jzeFQaCV4w644pzQJzVWay2zcA,2016,132,44,0.000244,1,0,1,100000000.0,princetonLocky
2,112536im7hy6wtKbpH1qYDWtTyMRAcA2p7,2016,246,0,1.000000,1,0,2,200000000.0,princetonCerber
3,1126eDRw2wqSkWosjTCre8cjjQW8sSeWH7,2016,322,72,0.003906,1,0,2,71200000.0,princetonCerber
4,1129TSjKtx65E35GiUo4AYVeyo48twbrGX,2016,238,144,0.072848,456,0,1,200000000.0,princetonLocky


In [9]:
df.sample(3)

,address,year,day,length,weight,count,looped,neighbors,income,label
82302,1J2qWfsMzb84way9BEPYfWzEYD4HAW4pY4,2011,51,54,0.000031,1,0,2,1.345000e+09,white
2500784,1Kf6Mgwpu6p9ZGSiA4ECJya57JEdjNQ37g,2017,280,20,0.250000,1,0,2,6.685126e+08,white
80783,18sUwRYvg1zqYggi7ykMo36TAh3qtZFdbo,2011,50,0,0.500000,1,0,2,2.687000e+09,white


In [10]:
df.tail(5)

,address,year,day,length,weight,count,looped,neighbors,income,label
2916692,12D3trgho1vJ4mGtWBRPyHdMJK96TRYSry,2018,330,0,0.111111,1,0,1,1.255809e+09,white
2916693,1P7PputTcVkhXBmXBvSD9MJ3UYPsiou1u2,2018,330,0,1.000000,1,0,1,4.409699e+07,white
2916694,1KYiKJEfdJtap9QX2v9BXJMpz2SfU4pgZw,2018,330,2,12.000000,6,6,35,2.398267e+09,white
2916695,15iPUJsRNZQZHmZZVwmQ63srsmughCXV4a,2018,330,0,0.500000,1,0,1,1.780427e+08,white
2916696,3LFFBxp15h9KSFtaw55np8eP5fv6kdK17e,2018,330,144,0.073972,6800,0,2,1.123500e+08,white


**🚨 Reflection 🚨**

The dataset under consideration contains approximately 3 million rows, a figure which has the potential to increase RAM usage and the time required to perform exploratory data analysis (EDA), generate visualisations, and test different machine learning models. Consequently, the utilisation of a representative sample of the data can expedite and optimise the analysis process, thereby facilitating the identification of patterns, distributions, and potential outliers. This reduction should, wherever possible, preserve the characteristics and proportions of the classes in the dataset, thus ensuring more representative results. Subsequent to this step, models can be evaluated using a sample, and, if necessary, trained on the full dataset to facilitate comparison of results.


### 🕵🏾 Exploratory Data Analysis (EDA)

The investigation phase. Here, we "dive" into the data to uncover patterns, anomalies, and financial insights. We use statistical summaries and visual charts (such as analyzing credit limit distributions, monthly payment track records, and the target class imbalance between Ransomware / Illicit Bitcoin Address ) to understand the dataset's characteristics before making any modifications or transformations.

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2916697 entries, 0 to 2916696
Data columns (total 10 columns):
 #   Column     Dtype  
---  ------     -----  
 0   address    str    
 1   year       int64  
 2   day        int64  
 3   length     int64  
 4   weight     float64
 5   count      int64  
 6   looped     int64  
 7   neighbors  int64  
 8   income     float64
 9   label      str    
dtypes: float64(2), int64(6), str(2)
memory usage: 331.3 MB


In [12]:
df.isnull().sum()

address      0
year         0
day          0
length       0
weight       0
count        0
looped       0
neighbors    0
income       0
label        0
dtype: int64

**⚠⚠ Important: ⚠⚠**

I would not automatically remove repeated rows. The same Bitcoin address may appear across different time windows, and this does not necessarily mean that the records are improper duplicates.

In [13]:
df.duplicated().sum()

np.int64(0)

In [14]:
df.nunique()

address      2631095
year               8
day              365
length            73
weight        784927
count          11572
looped         10168
neighbors        814
income       1866365
label             29
dtype: int64

The interval of days must account for leap years; the dataset records the day of the year as a value between 1 and 365, so any instance of 366 should be investigated in light of the dataset's definition and version.

In [15]:
print(f"Year range: {df["year"].min()} ~ {df["year"].max()}")
print(f"Day range: {df["day"].min()} ~ {df["day"].max()}")

print("Invalid day values:",
      (~df["day"].between(1, 366)).sum())

Year range: 2011 ~ 2018
Day range: 1 ~ 365
Invalid day values: 0


In [16]:
numeric_cols = df.select_dtypes(
    include="number"
).columns.tolist()

categorical_cols = df.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print(f"Numeric columns: {numeric_cols}")
print(f"Categorical columns: {categorical_cols}")

Numeric columns: ['year', 'day', 'length', 'weight', 'count', 'looped', 'neighbors', 'income']
Categorical columns: ['address', 'label']


In [17]:
df[numeric_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
year,2916697.0,2.014475e+03,2.257398e+00,2.011000e+03,2.013000e+03,2.014000e+03,2.016000e+03,2.018000e+03
day,2916697.0,1.814572e+02,1.040118e+02,1.000000e+00,9.200000e+01,1.810000e+02,2.710000e+02,3.650000e+02
length,2916697.0,4.500859e+01,5.898236e+01,0.000000e+00,2.000000e+00,8.000000e+00,1.080000e+02,1.440000e+02
weight,2916697.0,5.455192e-01,3.674255e+00,3.606469e-94,2.148438e-02,2.500000e-01,8.819482e-01,1.943749e+03
count,2916697.0,7.216446e+02,1.689676e+03,1.000000e+00,1.000000e+00,1.000000e+00,5.600000e+01,1.449700e+04
looped,2916697.0,2.385067e+02,9.663217e+02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.449600e+04
neighbors,2916697.0,2.206516e+00,1.791877e+01,1.000000e+00,1.000000e+00,2.000000e+00,2.000000e+00,1.292000e+04
income,2916697.0,4.464889e+09,1.626860e+11,3.000000e+07,7.428559e+07,1.999985e+08,9.940000e+08,4.996440e+13


In [18]:
df[categorical_cols].describe().T

,count,unique,top,freq
address,2916697,2631095,1LXrSb67EaH1LGc6d6kWHq8rgv4ZBQAcpU,420
label,2916697,29,white,2875284


In [19]:
df_analysis = df.copy()
df_analysis["label"] = np.where(df["label"] == "white", 0, 1)
df_analysis['label_v1'] = df['label'].apply(
    lambda x: "Lawful" if x == 'Lawful' else "Illicit Bitcoin Address"
)

In [20]:
plot_cols = [
    "length",
    "weight",
    "count",
    "looped",
    "neighbors",
    "income"
]

#### Auxiliary Functions

In [21]:
def F_barplot(data:pd.DataFrame, column:str):

    year_counts = data[column].value_counts().sort_index()
    
    sns.set_theme(style="whitegrid", context="notebook")
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    sns.barplot(
        x=year_counts.index,
        y=year_counts.values,
        color="steelblue",
        ax=ax
    )
    
    ax.set_title(
        f"Number of Records per {column}",
        fontsize = 16,
        fontweight = "bold",
        pad = 15
    )
    ax.set_xlabel(column, fontsize=12, labelpad=10)
    ax.set_ylabel("Number of Records", fontsize=12, labelpad=10)
    
    ax.tick_params(axis="x", rotation=45, labelsize=10)
    ax.tick_params(axis="y", labelsize=10)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.grid(axis="x", visible=False)
    
    sns.despine()
    
    ax.bar_label(
        ax.containers[0],
        fmt = "%.0f",
        padding = 3,
        fontsize = 9
    )
    
    plt.tight_layout()
    plt.show()

In [22]:
def F_lineplot(data:pd.DataFrame, column:str):
    day_counts = data[column].value_counts().sort_index()
    
    plt.figure(figsize=(12, 5))
    sns.lineplot(x=day_counts.index, y=day_counts.values)
    
    plt.title(f"Number of Records per {column} of Year")
    plt.xlabel("Day of Year")
    plt.ylabel("Number of Records")
    plt.tight_layout()
    plt.show()

In [23]:
def F_lineplot_label_comparison(data:pd.DataFrame, column:str):
    day_counts = (
        data.groupby([column, "label"])
          .size()
          .reset_index(name="records")
    )
    
    day_counts["Transaction Type"] = day_counts["label"].map({
        0: "Lawful",
        1: "Illicit Bitcoin Address"
    })

    sns.set_theme(style="whitegrid", context="notebook")
    
    fig, ax = plt.subplots(figsize=(14, 6))
    
    sns.lineplot(
        data = day_counts,
        x = column,
        y = "records",
        hue = "Transaction Type",
        hue_order = ["Lawful", "Illicit Bitcoin Address"],
        palette = {"Lawful": "steelblue", "Illicit Bitcoin Address": "indianred"},
        marker = "o",
        markersize = 4,
        linewidth = 2,
        ax = ax
    )
    
    ax.set_title(
        "Number of Transactions per Day of Year",
        fontsize = 16,
        fontweight = "bold",
        pad = 15
    )
    ax.set_xlabel("Day of Year", fontsize=12, labelpad=10)
    ax.set_ylabel("Number of Records", fontsize=12, labelpad=10)
    
    ax.legend(
        title = "Transaction Type",
        frameon = True
    )
    
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.grid(axis="x", visible=False)
    
    sns.despine()
    plt.tight_layout()
    plt.show()

In [24]:
def F_barplot_label_comparison(data:pd.DataFrame):

    label_counts = data["label"].value_counts().sort_index()
    label_percentages = label_counts / label_counts.sum() * 100
    
    sns.set_theme(style="whitegrid", context="notebook")
    
    fig, ax = plt.subplots(figsize=(9, 6))
    
    sns.barplot(
        x = label_counts.index,
        y = label_counts.values,
        hue = label_counts.index,
        palette = {0: "steelblue", 1: "indianred"},
        legend = False,
        ax = ax
    )
    
    ax.set_title("Target Class Distribution", fontsize = 16, fontweight = "bold", pad = 15)
    ax.set_xlabel("Label", fontsize = 12, labelpad = 10)
    ax.set_ylabel("Number of Records", fontsize = 12, labelpad = 10)
    
    for i, (count, percentage) in enumerate(
        zip(label_counts.values, label_percentages.values)
    ):
        ax.annotate(
            f"{count:,.0f}\n({percentage:.2f}%)",
            xy = (i, count),
            xytext = (0, 6),
            textcoords = "offset points",
            ha = "center",
            fontsize = 10,
            fontweight = "bold"
        )
    
    ax.grid(axis = "y", linestyle = "--", alpha = 0.4)
    ax.grid(axis = "x", visible = False)
    sns.despine()
    
    plt.tight_layout()
    plt.show()

In [25]:
def F_get_sample(data:pd.DataFrame, number:int = 200_000, random_state:int = 42) -> pd.DataFrame:
    return data.sample(
        n = min(number, len(data)),
        random_state = random_state
    )

In [26]:
def F_histplot(data:pd.DataFrame):

    
    fig, axes = plt.subplots(
        nrows = 3,
        ncols = 2,
        figsize = (14, 14)
    )
    
    for col, ax in zip(plot_cols, axes.flatten()):
        sns.histplot(
            data = F_get_sample(data),
            x = col,
            bins = 40,
            ax = ax
        )
    
        ax.set_title(f"Distribution of {col}")
    
    plt.tight_layout()
    plt.show()

In [27]:
def F_outlier_analysis(data:pd.DataFrame, plot_cols:list[str]):
    fig, axes = plt.subplots(
        nrows = 3,
        ncols = 2,
        figsize =(14, 12)
    )
    
    for col, ax in zip(plot_cols, axes.flatten()):
        sns.boxplot(
            data = F_get_sample(data),
            x = col,
            ax = ax
        )
    
        ax.set_title(f"Outlier Analysis: {col}")
    
    plt.tight_layout()
    plt.show()

In [28]:
def F_baxplot_label_comparison(data:pd.DataFrame, plot_cols:list[str]):
    fig, axes = plt.subplots(
        nrows = 3,
        ncols = 2,
        figsize = (14, 14)
    )
    
    for col, ax in zip(plot_cols, axes.flatten()):
        sns.boxplot(
            data = F_get_sample(data),
            hue = column,
            palette = "ch:s=.25,rot=-.25",
            x = "label_v1",
            y = col,
            ax = ax
        )
    
        ax.set_title(f"{col} by Label")
    
    plt.tight_layout()
    plt.show()

In [29]:
def F_correlation_matrix(data:pd.DataFrame):
    correlation_matrix = data.select_dtypes(include=['float64', 'int64']).corr()
    
    plt.figure(figsize = (8, 8))
    sns.heatmap(correlation_matrix, annot = True, fmt = '.2f', cmap = 'coolwarm', square = True, cbar_kws = {"shrink": .8})
    plt.title('Heatmap of Correlation Matrix for Bitcoin Heist Ransomware Address')
    plt.show()

#### Dashboards

In [30]:
# F_barplot(df_analysis, "year")

In [31]:
# F_lineplot(df_analysis, "day")

In [32]:
# F_lineplot_label_comparison(df_analysis, "day")

In [33]:
# label_counts = df_analysis["label"].value_counts(dropna=False)
# label_percent = df_analysis["label"].value_counts(normalize=True, dropna=False) * 100

# label_summary = pd.DataFrame({
#     "count": label_counts,
#     "percentage": label_percent.round(2)
# })

# display(label_summary)

In [34]:
# F_barplot_label_comparison(df_analysis)

In [35]:
# F_histplot(df_analysis)

In [36]:
# F_outlier_analysis(df_analysis, plot_cols)

Observations outside the traditional IQR limits

This calculation serves to flag extreme values, not to automatically determine that they are errors. In the context of ransomware, extreme values ​​may well be the very cases of greatest analytical interest.

In [37]:
for col in plot_cols:
    q1 = df_analysis[col].quantile(0.25)
    q3 = df_analysis[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = (
        (df_analysis[col] < lower) |
        (df_analysis[col] > upper)
    )

    print(
        f"{col}: {outliers.sum():,} "
        f"({outliers.mean() * 100:.2f}%)"
    )

length: 0 (0.00%)
weight: 55,006 (1.89%)
count: 688,188 (23.59%)
looped: 411,820 (14.12%)
neighbors: 114,659 (3.93%)
income: 430,677 (14.77%)


In [38]:
# F_baxplot_label_comparison(df_analysis, plot_cols)

In [39]:
# F_correlation_matrix(df_analysis)

⚠⚠ The goal is to determine whether there are repeated addresses across different temporal observations and identify whether the same address appears associated with different labels. This may reveal important characteristics regarding how the dataset was constructed, which should perhaps be understood before splitting the data into training and test sets. ⚠⚠

In [40]:
print(f"Unique addresses: {df_analysis["address"].nunique()}")
print(f"Total records: {len(df_analysis)}")

print(
    df.groupby("address", sort=False)
      .size()
      .describe().round(3)
)

Unique addresses: 2631095
Total records: 2916697
count    2631095.000
mean           1.109
std            1.212
min            1.000
25%            1.000
50%            1.000
75%            1.000
max          420.000
dtype: float64


### ⚙ Pre-processing.

**The beginning of everything**

Correlation analysis was initially used to identify relationships between features and the target variable and to detect highly correlated features. Feature selection was then performed to reduce redundancy and identify the most relevant predictors.

In [ ]:
# del df_analysis
# df_transf = df.copy()
# df_transf["label"] = np.where(df["label"] == "white", 0, 1)

In [ ]:
# df_minority = df_transf[df_transf['label'] == 1]
# df_majority = F_get_sample(
#     data = df_transf[df_transf['label'] == 0],
#     number = 300_000,
#     random_state = 42
# )

# df_balanced = (
#     pd.concat([df_minority, df_majority])
#     .sample(frac=1, random_state=42)
#     .reset_index(drop=True)
# )

# df_balanced['label'].value_counts()

In [ ]:
# X = df_balanced.loc[:, ~df_balanced.columns.isin(["address", "year", "day", "label"])].to_numpy()
# y = df_balanced['label'].to_numpy()

In [ ]:
X.shape
y.shape

## Second Semester:

**🤷🏾‍♂️ What will I do?**
* **Feature Engineering**
* **Predictive Modeling (Random Forest and XGBoost)**
* **Hyperparameter Tuning (`GridSearchCV`)**
* **Metric Evaluation (Recall, ROC-AUC)**
* **Dashboard Development**
* **Final Writing of the Capstone Report.**

### 🛠 Data Cleaning and Transformation 

In this step, we correct the issues identified during EDA. This includes handling missing values ​​(for example, by removing the corresponding rows), ensuring the data is in the correct format, and applying the function.

**Create a list of all columns with a ratio above 55% relative to the  [ default payment next month ] column [ if it exists ].**

In [ ]:
# correlation = df.corr(numeric_only=True)

# corr_list = []

# for column, value in zip(
#     correlation.columns,
#     correlation["default payment next month"].sort_values(ascending=False)
# ):
#     print(value)
#     # if value >= 0.55 and column not in ["ID", "default payment next month"]:
#     #     corr_list.append(column)

In [ ]:
df.columns

### ✂ Splitting into Training and Test Data

An essential step for model training and evaluation. We split the dataset into two parts: a larger one (training), which the model uses to learn, and a smaller one (test), which is kept "hidden" from the model to simulate new data and evaluate its actual performance after training.

**As stated in the document, the data needs to be split into the following formats [ 5%, 15%, and 25% ], so we will use a function to avoid code duplication.**

In [ ]:
def splittint_data(data:pd.DataFrame, label:pd.DataFrame, _test_size:float = 0.30, _random_state:int = 1, ):
    print(\
        f"""
             X.shape, y.shape, X_train.shape, X_test.shape, y_train.shape, y_test.shape
        """)
    return train_test_split(data, label, test_size = _test_size, random_state = _random_state, stratify=label)

### 👨🏽‍🔬 Predictive Modeling Pipeline

Here, we build an automated production pipeline for our model. The Scikit-learn Pipeline encapsulates all preprocessing stepS and the final model (Supervised Classification Models), ensuring that the same steps are consistently applied to both the training data and new data.


**Initially, we will use the three models below.**

* [Random Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
* [Decision Tree](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
* [Xgboost](https://xgboost.readthedocs.io/en/stable/parameter.html)

#### Building Xgboost Model

In [ ]:
import xgboost as xgb

#### Building Decision Tree Model

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
# clf = DecisionTreeClassifier(criterion='gini', max_depth = 4, random_state = 0)

# # Train Decision Tree Classifer
# clf = clf.fit(X_train, y_train)

# #Predict the response for test dataset
# y_pred = clf.predict(X_test)

In [ ]:
# # Model Accuracy, how often is the classifier correct?
# print("Accuracy:",metrics.accuracy_score(y_test, y_pred))

# # Rounded upto 2 decimal places
# print( "Accuracy: {:.2f}".format(metrics.accuracy_score(y_test, y_pred)) )

In [ ]:
# # Predict by using an object classifier 'clf'
# y_predict = clf.predict(X_test)

# # Calculate cm by using a method named as 'confusion_matrix'
# cm = confusion_matrix(y_test, y_predict)

# # Plot confusion matrix with integers using seaborn heatmap
# plt.figure(figsize=(6, 4))
# sns.heatmap(cm, annot=True, fmt='d')  # 'd' stands for integer format
# plt.xlabel('Predicted labels')
# plt.ylabel('True labels')
# plt.title('Confusion Matrix')
# plt.show()

# # Generate classification report
# report = classification_report(y_test, y_predict)
# print("Classification Report:")
# print(report)

In [ ]:
# **3 Splits de Treino (5%, 15% e 25%):**

In [ ]:
# from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# # Confusion Matrix
# cm = confusion_matrix(y_test, y_pred)
# disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Class 0", "Class 1"])
# disp.plot(cmap="Blues")
# plt.title(f"Confusion Matrix (k={best_k})")
# plt.grid(False)
# plt.show()

# # Detailed classification report
# print("Classification Report:")
# print(classification_report(y_test, y_pred, target_names=["Class 0", "Class 1"]))

#### Building Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

In [ ]:
# from sklearn.ensemble import RandomForestClassifier

# X, y = make_classification(
#     n_samples = 1000,
#     n_features = 4,
#     n_informative = 2,
#     n_redundant = 0,
#     random_state = 0,
#     shuffle = False
# )

# clf = RandomForestClassifier(max_depth=2, random_state=0)
# clf.fit(X, y)

# clf.predict([[0, 0, 0, 0]])

### 👨🏽‍🔬 Model Evaluation and Metric Interpretation

### 👨🏽‍🏫 Final Writing of the Capstone Report.